In [0]:
%pip install ftfy>=6.3 mammoth>=1.7 resiliparse>=0.14 pypdfium2>=4.30 loky
%pip install striprtf ocflzw-decompress  # existing deps, ensure installed
%pip install pytesseract Pillow PyMuPDF  # OCR cascade deps (tesseract binary provided by cluster init script)
dbutils.library.restartPython()

In [0]:

# Pre-flight: fail fast if cluster lacks required system binaries or Python libs.
import subprocess
_preflight_errors = []
try:
    subprocess.run(["tesseract", "--version"], check=True, capture_output=True, timeout=10)
except Exception as e:
    _preflight_errors.append(f"tesseract missing or broken: {e}")
for mod in ("ftfy", "mammoth", "resiliparse", "pypdfium2", "striprtf"):
    try:
        __import__(mod)
    except Exception as e:
        _preflight_errors.append(f"cannot import {mod}: {e}")
if _preflight_errors:
    raise RuntimeError("Pre-flight failed:\n" + "\n".join(_preflight_errors))
print("Pre-flight OK")


In [0]:
# ==================== SHARED DECODER (factored out) ====================
# Constants (OCF/PDF/TIFF markers, VERSION constants, MAX_BLOB_SIZE, BLOB_LENGTH_TOLERANCE),
# raw_content_sha256, and the full OCF-LZW decode cascade now live in ./blob_decoder so the
# text pipeline and Blob 4:Files share ONE decoder. The %run cell below defines those names;
# everything after (mime/pdf/docx/parse/post_process) is unchanged and relies on them.

In [0]:
%run ./blob_decoder

In [0]:

# ==================== MIME DETECTION ====================

try:
    import magic as _magic
    _HAVE_MAGIC = True
except Exception:
    _HAVE_MAGIC = False

try:
    import olefile as _olefile
    _HAVE_OLE = True
except Exception:
    _HAVE_OLE = False


def detect_mime(content: bytes) -> str:
    if not content:
        return 'application/octet-stream'
    if content.startswith(PDF_MAGIC):
        return 'application/pdf'
    if content.startswith(ZIP_MAGIC):
        head = content[:4096]
        if b'word/' in head:
            return 'application/vnd.openxmlformats-officedocument.wordprocessingml.document'
        if b'xl/' in head:
            return 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'
        if b'ppt/' in head:
            return 'application/vnd.openxmlformats-officedocument.presentationml.presentation'
        return 'application/zip'
    if content.startswith(OLE_MAGIC):
        return _classify_ole(content)
    if content.startswith(RTF_MAGIC) or content.startswith(RTF_MAGIC_LOOSE):
        return 'text/rtf'
    if _HAVE_MAGIC:
        try:
            return _magic.Magic(mime=True).from_buffer(content) or 'application/octet-stream'
        except Exception:
            pass
    return 'application/octet-stream'


def _classify_ole(data: bytes) -> str:
    if not _HAVE_OLE:
        return 'application/x-ole-storage'
    try:
        with _olefile.OleFileIO(io.BytesIO(data)) as ole:
            if ole.exists('WordDocument'):
                return 'application/msword'
            if ole.exists('Workbook') or ole.exists('Book'):
                return 'application/vnd.ms-excel'
            if ole.exists('PowerPoint Document'):
                return 'application/vnd.ms-powerpoint'
            return 'application/x-ole-storage'
    except Exception:
        return 'application/x-ole-storage'


In [0]:

# ==================== PDF EXTRACTION ====================

try:
    import pypdfium2 as _pdfium
    _HAVE_PDFIUM = True
except Exception:
    _HAVE_PDFIUM = False

try:
    import fitz as _fitz  # PyMuPDF
    _HAVE_FITZ = True
except Exception:
    _HAVE_FITZ = False

try:
    import pytesseract as _pytess
    from PIL import Image as _PILImage
    _HAVE_TESS = True
except Exception:
    _HAVE_TESS = False


def _extract_pdf_pdfium(content: bytes) -> Optional[str]:
    if not _HAVE_PDFIUM:
        return None
    pdf = None
    try:
        pdf = _pdfium.PdfDocument(io.BytesIO(content))
        parts = []
        try:
            for i in range(len(pdf)):
                page = pdf[i]
                tp = None
                try:
                    tp = page.get_textpage()
                    try:
                        t = tp.get_text_range() or ""
                    finally:
                        if tp is not None:
                            try:
                                tp.close()
                            except Exception:
                                pass
                finally:
                    try:
                        page.close()
                    except Exception:
                        pass
                if t.strip():
                    parts.append(t)
        finally:
            # Handled by outer finally below via `pdf.close()`; pass here.
            pass
        return "\n".join(parts) if parts else None
    except Exception:
        return None
    finally:
        if pdf is not None:
            try:
                pdf.close()
            except Exception:
                pass


def _extract_pdf_pymupdf(content: bytes) -> Optional[str]:
    if not _HAVE_FITZ:
        return None
    doc = None
    try:
        doc = _fitz.open(stream=content, filetype="pdf")
        if doc.needs_pass:
            try:
                doc.authenticate("")
            except Exception:
                pass
        parts = []
        for page in doc:
            t = page.get_text("text") or ""
            if t.strip():
                parts.append(t)
        return "\n".join(parts) if parts else None
    except Exception:
        return None
    finally:
        if doc is not None:
            try:
                doc.close()
            except Exception:
                pass


def _extract_pdf_ocr(content: bytes, max_pages: int = OCR_MAX_PAGES, dpi: int = 150) -> Optional[str]:
    if not (_HAVE_FITZ and _HAVE_TESS):
        return None
    if len(content) > OCR_MAX_PDF_SIZE_MB * 1024 * 1024:
        return None
    doc = None
    try:
        doc = _fitz.open(stream=content, filetype="pdf")
        parts = []
        for i, page in enumerate(doc):
            if i >= max_pages:
                break
            try:
                mat = _fitz.Matrix(dpi / 72, dpi / 72)
                pix = page.get_pixmap(matrix=mat)
                img_data = pix.tobytes("png")
                img = _PILImage.open(io.BytesIO(img_data))
                try:
                    # pytesseract's timeout= forwards to the tesseract subprocess
                    # and actually kills it; prevents pathological PDFs from hanging.
                    page_text = _pytess.image_to_string(
                        img, lang=OCR_LANG, timeout=OCR_PAGE_TIMEOUT_SEC
                    )
                except RuntimeError:
                    # pytesseract raises RuntimeError when the tesseract
                    # subprocess exceeds timeout
                    continue  # skip this page, move to the next
                if page_text and page_text.strip():
                    parts.append(page_text.strip())
            except Exception:
                continue
        return "\n\n".join(parts) if parts else None
    except Exception:
        return None
    finally:
        if doc is not None:
            try:
                doc.close()
            except Exception:
                pass


In [0]:

# ==================== DOCX / HTML / RTF / EXCEL ====================

try:
    import mammoth as _mammoth
    _HAVE_MAMMOTH = True
except Exception:
    _HAVE_MAMMOTH = False

try:
    import docx2txt as _docx2txt
    _HAVE_DOCX2TXT = True
except Exception:
    _HAVE_DOCX2TXT = False

try:
    from resiliparse.parse.html import HTMLTree as _RespHTMLTree
    from resiliparse.extract.html2text import extract_plain_text as _resp_extract
    _HAVE_RESILIPARSE = True
except Exception:
    _HAVE_RESILIPARSE = False

try:
    from striprtf.striprtf import rtf_to_text as _rtf_to_text
    _HAVE_STRIPRTF = True
except Exception:
    _HAVE_STRIPRTF = False

try:
    from openpyxl import load_workbook as _openpyxl_load
    _HAVE_OPENPYXL = True
except Exception:
    _HAVE_OPENPYXL = False


def _extract_docx(content: bytes) -> Optional[str]:
    # Primary: mammoth (preserves headings/bullets/tables)
    if _HAVE_MAMMOTH:
        try:
            res = _mammoth.convert_to_markdown(io.BytesIO(content))
            if res.value and res.value.strip():
                return res.value
        except Exception:
            pass
    # Fallback: docx2txt
    if _HAVE_DOCX2TXT:
        try:
            return _docx2txt.process(io.BytesIO(content))
        except Exception:
            pass
    return None


def _extract_html(content: bytes) -> Optional[str]:
    if not _HAVE_RESILIPARSE:
        # Fall back to BS4 (v1 behaviour) if resiliparse is missing.
        return _extract_html_bs4(content)
    for enc in ('utf-8', 'windows-1252', 'latin-1'):
        try:
            html = content.decode(enc, errors='ignore')
            if '<' not in html or '>' not in html:
                continue
            tree = _RespHTMLTree.parse(html)
            text = _resp_extract(tree, preserve_formatting='minimal_html', main_content=False)
            if text and text.strip():
                return text
        except Exception:
            continue
    return None


def _extract_html_bs4(content: bytes) -> Optional[str]:
    try:
        from bs4 import BeautifulSoup
    except Exception:
        return None
    for enc in ('utf-8', 'windows-1252', 'latin-1'):
        try:
            html = content.decode(enc, errors='ignore')
            if '<' not in html or '>' not in html:
                continue
            soup = BeautifulSoup(html, 'html.parser')
            for element in soup(['script', 'style', 'head', 'title', 'meta']):
                element.decompose()
            text = soup.get_text(separator=' ', strip=True)
            if text:
                return text
        except Exception:
            continue
    return None


def _extract_rtf(content: bytes) -> Optional[str]:
    if not (_HAVE_STRIPRTF and content):
        return None
    # Report: Cerner RTF is always cp1252. Skip UTF-8 trial.
    try:
        rtf_text = content.decode('cp1252', errors='ignore')
        if not (rtf_text.startswith('{\\rtf') or rtf_text.startswith('{\\')):
            # Second chance: utf-8 for non-Cerner RTFs
            rtf_text = content.decode('utf-8', errors='ignore')
            if not (rtf_text.startswith('{\\rtf') or rtf_text.startswith('{\\')):
                return None
        plain = _rtf_to_text(rtf_text, errors='ignore')
        return plain if plain and plain.strip() else None
    except Exception:
        return None


def _extract_excel(content: bytes) -> Optional[str]:
    if not _HAVE_OPENPYXL:
        return None
    try:
        wb = _openpyxl_load(io.BytesIO(content), read_only=True, data_only=True)
        parts = []
        for sheet in wb.sheetnames:
            ws = wb[sheet]
            for row in ws.iter_rows(values_only=True):
                row_text = ' '.join(str(c) for c in row if c is not None)
                if row_text.strip():
                    parts.append(row_text)
        return '\n'.join(parts) if parts else None
    except Exception:
        return None


def _guess_text(content: bytes) -> Tuple[Optional[str], Optional[str]]:
    # Lightweight encoding detect; report recommends short-circuiting known formats.
    # Here we only handle genuinely unknown binary-ish content.
    if not content:
        return None, None
    # BOM checks — decode strictly; BOM is a strong signal and a UnicodeDecodeError
    # here means the stream is genuinely malformed for that encoding, so fall through.
    if content.startswith(b'\xff\xfe'):
        try:
            return content.decode('utf-16-le'), 'utf-16-le'
        except UnicodeDecodeError:
            pass
    if content.startswith(b'\xfe\xff'):
        try:
            return content.decode('utf-16-be'), 'utf-16-be'
        except UnicodeDecodeError:
            pass
    if content.startswith(b'\xef\xbb\xbf'):
        try:
            return content[3:].decode('utf-8'), 'utf-8-sig'
        except UnicodeDecodeError:
            pass
    # Strict utf-8: only succeeds on well-formed UTF-8. Falls through on invalid
    # byte sequences so cp1252/latin-1 get a real chance (the previous
    # errors='ignore' path always succeeded and made this cascade dead code).
    try:
        return content.decode('utf-8'), 'utf-8'
    except UnicodeDecodeError:
        pass
    # Strict cp1252: almost all byte sequences decode, but some (0x81/0x8D/0x8F/
    # 0x90/0x9D) are undefined and raise — fall through to latin-1 replace.
    try:
        return content.decode('cp1252'), 'cp1252'
    except UnicodeDecodeError:
        pass
    # latin-1 (ISO-8859-1) can't fail (every byte is a valid code point), but we
    # use errors='replace' so any truly undecodable points would surface as U+FFFD
    # rather than silently disappear.
    return content.decode('latin-1', errors='replace'), 'latin-1'


In [0]:

# ==================== PARSE DISPATCHER ====================

def parse(content: bytes, provided_type=None) -> Tuple[Optional[str], Optional[str], Optional[str]]:
    """Returns (text, content_type, encoding).
    For binary content types that are not text, returns (None, content_type, None).
    """
    if not content:
        return None, None, None
    content_type = provided_type or detect_mime(content)

    if content_type in BINARY_CONTENT_TYPES:
        return None, content_type, None

    if content_type == 'application/pdf':
        t = _extract_pdf_pdfium(content) or _extract_pdf_pymupdf(content)
        if t:
            return t, content_type, 'utf-8'
        t = _extract_pdf_ocr(content)
        if t:
            return t, content_type, 'utf-8'
        # Give a concrete status string so callers don't mistake this for a crash
        return "[PDF - extraction failed]", content_type, None

    if content_type == 'application/vnd.openxmlformats-officedocument.wordprocessingml.document':
        t = _extract_docx(content)
        if t:
            return t, content_type, 'utf-8'
        # DOCX is a ZIP container — do NOT fall through to _guess_text on the
        # raw ZIP bytes or we'd produce garbled "text" that looks legitimate.
        return "[DOCX - extraction failed]", content_type, 'utf-8'

    if content_type in ('application/vnd.ms-excel',
                        'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'):
        t = _extract_excel(content)
        if t:
            return t, content_type, 'utf-8'
        # xlsx is a ZIP, xls is OLE2 — both binary containers; never _guess_text.
        return "[Excel - extraction failed]", content_type, 'utf-8'

    if content_type in ('text/html', 'text/xml', 'application/xhtml+xml'):
        t = _extract_html(content)
        if t:
            return t, content_type, 'utf-8'
        # Both resiliparse and BS4 extractors failed; don't _guess_text raw HTML.
        return "[HTML - extraction failed]", content_type, 'utf-8'

    if content_type == 'text/rtf' or content.startswith(RTF_MAGIC) or content.startswith(RTF_MAGIC_LOOSE):
        t = _extract_rtf(content)
        if t:
            return t, (content_type or 'text/rtf'), 'utf-8'
        # Consistent with Fix 2 pattern: structured RTF that couldn't be parsed
        # becomes a failure marker rather than falling through to _guess_text,
        # which would spit out raw RTF control words as "text".
        return "[RTF - extraction failed]", (content_type or 'text/rtf'), 'utf-8'

    # Plaintext / unknown
    decoded, enc = _guess_text(content)
    if decoded:
        return decoded, content_type, enc

    return "[Binary data, unable to decode]", content_type, None


In [0]:

# ==================== PARSE HANG DEFENSE ====================
# The OCR subprocess timeout (OCR_PAGE_TIMEOUT_SEC) only covers tesseract; the
# pypdfium2 / PyMuPDF primary text extraction paths can themselves hang on
# pathological PDFs with no Python-level timeout API. Size cap short-circuits
# the worst cases (GB-scale decompression DoS) and the subprocess timeout
# enforces a real wall-clock limit on the whole parse() call.

# Use loky's ProcessPoolExecutor (not concurrent.futures') — loky serializes
# submitted tasks via cloudpickle, which correctly handles functions defined
# in %run-included notebooks (where stdlib pickle fails with PicklingError).
# Loky also defaults to spawn-mode subprocess, which is safer than fork inside
# a PySpark Python worker (avoids inheriting JVM-bridge/MKL/OpenMP state).
from loky import ProcessPoolExecutor
from concurrent.futures import TimeoutError as _FutureTimeout


def _is_probable_pdf(content: bytes, provided_type: Optional[str]) -> bool:
    if provided_type and "pdf" in provided_type.lower():
        return True
    return bool(content) and content[:4] == b"%PDF"


class ParsePool:
    """Reusable subprocess pool for bounded-time parse() calls.

    Usage inside an executor pandas UDF:
        with ParsePool() as pool:
            for row in ...:
                text, ct, enc, parse_status = pool.parse(content, provided_type)

    On timeout the pool is torn down and recreated (the worker subprocess may
    still be running native C code that can't be interrupted Python-side).
    Uses loky.ProcessPoolExecutor internally (cloudpickle-aware, spawn-mode
    default). Safe to construct inside a PySpark Python worker.
    """

    def __init__(self, timeout: int = PARSE_TIMEOUT_SEC):
        self._timeout = timeout
        self._pool: Optional[ProcessPoolExecutor] = None

    def __enter__(self):
        self._pool = ProcessPoolExecutor(max_workers=1)
        return self

    def __exit__(self, *a):
        self._teardown()

    def _teardown(self):
        if self._pool is not None:
            try:
                self._pool.shutdown(wait=False, cancel_futures=True)
            except Exception:
                pass
            self._pool = None

    def _recreate(self):
        self._teardown()
        self._pool = ProcessPoolExecutor(max_workers=1)

    def parse(self, content: bytes, provided_type: Optional[str] = None):
        """Returns (text, content_type, encoding, parse_status)
        where parse_status is one of: "ok", "pdf_too_large", "timeout", "error:<class>".
        """
        if _is_probable_pdf(content, provided_type) and len(content) > PDF_EXTRACT_MAX_BYTES:
            return "[PDF - too large]", "application/pdf", "utf-8", "pdf_too_large"
        if self._pool is None:
            self._pool = ProcessPoolExecutor(max_workers=1)
        try:
            fut = self._pool.submit(parse, content, provided_type)
            text, ct, enc = fut.result(timeout=self._timeout)
            return text, ct, enc, "ok"
        except _FutureTimeout:
            self._recreate()
            return ("[PARSE - timeout]", provided_type or "application/octet-stream",
                    "utf-8", "timeout")
        except Exception as e:
            # Runtime error inside the child; pool may be poisoned
            self._recreate()
            return (f"[PARSE - error: {type(e).__name__}]",
                    provided_type or "application/octet-stream",
                    "utf-8", f"error:{type(e).__name__}")


def parse_with_timeout(content: bytes, provided_type: Optional[str] = None,
                       timeout: int = PARSE_TIMEOUT_SEC):
    """Single-shot convenience wrapper (creates and tears down a pool).

    Prefer ParsePool for batches — avoids fork-per-row overhead.
    """
    with ParsePool(timeout=timeout) as pool:
        return pool.parse(content, provided_type)


In [0]:

# ==================== POST-PROCESSOR ====================

import ftfy as _ftfy
from ftfy import TextFixerConfig as _FtfyConfig

_FTFY_CONFIG = _FtfyConfig(
    fix_c1_controls=True,
    restore_byte_a0=True,
    fix_line_breaks=True,
    normalization="NFC",
    fix_latin_ligatures=False,
)

# Garbage patterns carried over from v1
_GARBAGE_PATTERNS = [
    r'f{8,}',
    r'(f5ebe8){2,}',
    r'(bec7c2){2,}',
    r'(333333){2,}',
    r'(cdcdcd){2,}',
    r'(elnlueve){2,}',
    r'(7f){4,}',
]
_GARBAGE_REGEX = re.compile('|'.join(_GARBAGE_PATTERNS), re.IGNORECASE)
_OCF_TEXT_PATTERN = re.compile(r'\s*ocf_blob\s*', re.IGNORECASE)
_TEMPLATE_PATTERN = re.compile(r'<%[A-Z_]*%>', re.IGNORECASE)


def _strip_ocf_text(text: str) -> str:
    if not text or 'ocf_blob' not in text.lower():
        return text
    return _OCF_TEXT_PATTERN.sub('', text).strip()


def _strip_template(text: str) -> str:
    if not text or '<%' not in text:
        return text
    return _TEMPLATE_PATTERN.sub('', text).strip()


def _truncate_at_garbage(text: str) -> Optional[str]:
    if not text:
        return text
    m = _GARBAGE_REGEX.search(text)
    if not m:
        return text
    pos = m.start()
    if pos < 20:
        return None
    truncated = text[:pos].rstrip()
    truncated = re.sub(r'[0-9a-f]{6,}$', '', truncated, flags=re.IGNORECASE).rstrip()
    return truncated if len(truncated) >= 20 else None


def _detect_truncation(text: str, content_type: str, decompressed_bytes: bytes,
                       blob_length: Optional[int], decompressor_flag: bool) -> Tuple[bool, Optional[str]]:
    """Returns (truncation_flag, truncation_reason)."""
    if decompressor_flag:
        return True, "blob_length_mismatch"
    if content_type == 'text/rtf' and text:
        # Brace-count mismatch is the reliable RTF truncation signal. The
        # previous last-char check (flag if stripped end is not '}' or '.')
        # over-triggered on healthy RTF that ends with \par, a control word,
        # or plain text — after extraction those are all normal endings.
        if text.count('{') != text.count('}'):
            return True, "rtf_brace_mismatch"
    if content_type == 'application/pdf' and decompressed_bytes:
        tail = decompressed_bytes[-1024:]
        if b'%%EOF' not in tail:
            return True, "pdf_no_eof"
    if text and decompressed_bytes:
        dlen = len(decompressed_bytes)
        # Catch truncation on *both* sides of an 8192-byte boundary: LZW can
        # stop mid-code-block at e.g. 8190 or 16386 bytes, so accept remainder
        # within 64 of either edge. (abs(dlen % 8192) is redundant — modulo
        # is already non-negative.)
        if dlen > 8192:
            rem = dlen % 8192
            if rem < 64 or rem > 8128:
                last = text.rstrip()
                if last and last[-1].isalpha():
                    return True, "ends_mid_word_near_8192"
    return False, None


def post_process(text: str, content_type: str, decompressed_bytes: bytes,
                 blob_length: Optional[int], decompressor_flag: bool = False
                 ) -> Tuple[Optional[str], dict]:
    """Clean text; return (cleaned_text, metadata).

    metadata keys: ftfy_explain (str JSON | None), ftfy_failed (bool),
    truncation_flag (bool), truncation_reason (str | None)
    """
    meta = {"ftfy_explain": None, "ftfy_failed": False,
            "truncation_flag": False, "truncation_reason": None}
    if not isinstance(text, str) or not text:
        return text, meta

    # 1. OCF marker strip
    text = _strip_ocf_text(text)
    if not text:
        return None, meta

    # 2. Template marker strip
    text = _strip_template(text)
    if not text:
        return None, meta

    # 3. Garbage truncate (can return None if almost all garbage)
    text = _truncate_at_garbage(text)
    if not text:
        return None, meta

    # 4. ftfy with audit. Split the json.dumps serialization into its own
    # try/except so a non-serializable step can't silently bypass the fallback
    # or corrupt `text`. Any ftfy *or* serialization failure flips
    # meta["ftfy_failed"] = True so the regression is observable downstream.
    try:
        fixed, explanation = _ftfy.fix_and_explain(text, config=_FTFY_CONFIG)
        text = fixed
        if explanation:
            try:
                meta["ftfy_explain"] = json.dumps(
                    [(step.transformation, step.encoding) if hasattr(step, 'transformation') else str(step)
                     for step in explanation]
                )
            except Exception:
                meta["ftfy_failed"] = True
                meta["ftfy_explain"] = None
    except Exception:
        # Never block on ftfy; fall back to v1-style regex replacement
        meta["ftfy_failed"] = True
        for bad, good in [('Â ', ' '), ('â€™', "'"), ('Ã©', 'é')]:
            text = text.replace(bad, good)

    # 5. Final whitespace cleanup
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = text.strip()

    if len(text) < 1:
        return None, meta

    # 6. Truncation heuristic
    flag, reason = _detect_truncation(text, content_type, decompressed_bytes, blob_length, decompressor_flag)
    meta["truncation_flag"] = flag
    meta["truncation_reason"] = reason

    return text, meta


print(f"blob_shared_lib loaded: decompressor={DECOMPRESSOR_VERSION} parser={PARSER_VERSION} post_processor={POST_PROCESSOR_VERSION}")